# Module `algorithms/genetic.py`

Le squelette est un GA classique : **selection -> croisement -> mutation -> remplacement**. Ce qui change pour le VRP, c'est qu'on ne peut **pas** coder un individu comme une liste de tournees telle quelle. Si on essaie de croiser deux solutions VRP brutes (listes de listes de clients), un croisement classique a un point produit immediatement deux pathologies :

- **doublons / oublis** : un client peut apparaitre dans deux tournees ou disparaitre,
- **violation de capacite** : la coupure ne tient pas compte de la charge des sous-tournees.

La parade standard (Prins, 2004) est de manipuler une representation **plus simple** que la solution finale : une permutation de tous les clients, appelee *giant-tour*. Les operateurs de croisement et de mutation travaillent sur cette permutation, et un decodeur dedie (`_split`) la transforme en tournees admissibles **de cout minimal sous l'ordre fixe**.

Le pipeline complet d'une generation est donc :

1. **Selection** : tournoi a `k` participants (pression selective reglable).
2. **Croisement** : OX (Order Crossover), specifiquement concu pour les permutations.
3. **Mutation** : swap ou inversion d'un segment.
4. **Decodage** : `_split` (programmation dynamique O(n^2)) -> routes VRP.
5. **Education memetique optionnelle** : `local_search` sur les routes, pour que chaque individu soit deja un optimum local avant d'entrer dans la population.
6. **Remplacement generationnel** avec elitisme (les `elitism` meilleurs survivent inconditionnellement).

Les sections suivantes justifient chaque brique en montrant **ce qui casserait sans elle**.


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

from cesipath import GraphGenerationConfig, GraphGenerator
from cesipath.algorithms import genetic_algorithm
from cesipath.algorithms.genetic import _ox_crossover, _mutate, _split
from cesipath.solver_input import build_static_solver_input
import random


## 1. Instance de travail


In [2]:
cfg = GraphGenerationConfig(node_count=15, seed=4)
instance = GraphGenerator(cfg).generate()
si = build_static_solver_input(instance)
print(f"n = {instance.node_count}, capacite = {si.vehicle_capacity}")


n = 15, capacite = 40


## 2. Codage giant-tour + Split de Prins

**Pourquoi cette indirection ?** Imaginons qu'on code un individu directement comme `[[1,3,5],[2,4]]`. Croiser deux tels individus avec n'importe quel operateur generique donne, dans le cas general, une solution **invalide** (doublons, oublis, capacite cassee). Reparer un croisement par post-traitement (suppression des doublons, reinsertion des manquants) introduit un biais arbitraire et detruit l'information genetique transmise.

L'astuce de Prins [1] consiste a separer **la decision combinatoire** (l'ordre des clients) de **la decision logistique** (ou couper en sous-tournees) :

- Un *individu* = une permutation de tous les clients : `[14, 11, 1, 13, 7, 6, ...]`. C'est juste une liste, donc OX et les mutations classiques de TSP s'appliquent sans modification.
- Le *Split* prend cette permutation et calcule, par programmation dynamique, **le decoupage optimal en sous-tournees admissibles** sous la contrainte de capacite, **sans changer l'ordre des clients**.

Plus formellement, le Split construit un **graphe auxiliaire** (different du graphe routier !) dont les sommets sont les indices `0, 1, ..., n` de la permutation. On y place une arete orientee `i -> j` (toujours de gauche a droite, `j > i`) si servir les clients `giant_tour[i:j]` en une seule tournee respecte la capacite ; le poids de l'arete est le cout de cette sous-tournee (depot -> clients -> depot). Comme tous les arcs vont strictement vers la droite, il **ne peut pas y avoir de cycle** : on dit que ce graphe est un **DAG** (Directed Acyclic Graph = graphe oriente acyclique). Trouver le decoupage optimal revient alors a chercher le plus court chemin de `0` a `n` dans ce DAG.

Sur un DAG, le plus court chemin se calcule par programmation dynamique en parcourant les sommets dans l'ordre topologique (ici l'ordre naturel `0, 1, ..., n`) :

- `v[j]` = cout minimum pour servir les `j` premiers clients de la permutation,
- `p[j]` = indice du debut de la derniere sous-tournee dans le decoupage optimal.

Pour chaque debut `i`, on etend une route tant que la charge tient ; chaque extension met a jour `v[j+1]` si elle ameliore. La complexite annoncee est **O(n^2)** [1, section 3] : double parcours `(i, j)`, et la boucle interne s'arrete des que la capacite est depassee.

**Propriete cle** : pour une permutation donnee, le Split donne le **meilleur** decoupage possible (preuve dans [1, proposition 1]). La selection ne compare donc jamais des decoupages mediocres entre eux ; elle compare des permutations sur la base de leur meilleur decoupage. C'est ce qui rend le GA competitif.


In [3]:
depot = si.depot
clients = [c for c, d in si.demands.items() if c != depot and d > 0]
rng = random.Random(1)
perm = clients[:]
rng.shuffle(perm)

routes, cost = _split(perm, depot, si.cost_matrix, si.demands, si.vehicle_capacity)
print("giant-tour :", perm)
print(f"cout Split : {cost:.2f}  en {len(routes)} sous-tournees")
for r in routes:
    print("  ", r)


giant-tour : [14, 11, 1, 13, 7, 6, 4, 9, 8, 12, 5, 2, 10, 3]
cout Split : 1687.68  en 8 sous-tournees
   [0, 14, 0]
   [0, 11, 1, 0]
   [0, 13, 7, 0]
   [0, 6, 4, 0]
   [0, 9, 8, 0]
   [0, 12, 5, 0]
   [0, 2, 0]
   [0, 10, 3, 0]


## 3. OX crossover (Order Crossover)

**Pourquoi pas un croisement classique a un point ?** Sur des permutations, c'est interdit. Prenons deux parents :

```
p1 = [1, 2, 3, 4, 5]
p2 = [3, 5, 1, 4, 2]
```

Un croisement a un point (coupe apres l'indice 2) donne `[1, 2, 3, 4, 2]` : le client 2 est duplique, le 5 a disparu. Resultat : individu invalide.

OX (Order Crossover), introduit par Davis [2] pour les problemes d'ordonnancement, est concu pour preserver la propriete "permutation" **par construction** :

1. On choisit aleatoirement un segment `[i, j]` du parent 1 et on le copie tel quel dans l'enfant : ce bloc preserve un **sous-ordre absolu** de p1.
2. On parcourt p2 a partir de `j+1` (en cyclique) et on insere dans l'enfant, dans l'ordre rencontre, **uniquement les clients pas encore presents**.

Resultat : une permutation valide qui herite d'un bloc contigu de p1 et de l'**ordre relatif** des autres clients selon p2. Cette transmission combinee (sous-ordre absolu + ordre relatif) correspond aux **schemas** pertinents pour les problemes de tournees - terme issu du theoreme des schemas de Holland [3], qui designe en GA les motifs de gene partages que la selection est censee propager. Pour un VRP, ces motifs sont les **adjacences** (paires de clients consecutifs), puisque le cout depend directement des arcs empruntes.

Le test ci-dessous verifie l'invariant essentiel : `sorted(child) == sorted(clients)` (pas de doublon, pas d'oubli).


In [4]:
rng = random.Random(42)
p1 = clients[:]
p2 = clients[:]
rng.shuffle(p1)
rng.shuffle(p2)
child = _ox_crossover(p1, p2, rng)
print("p1    :", p1)
print("p2    :", p2)
print("child :", child)
print("meme multiset ?", sorted(child) == sorted(clients))


p1    : [9, 13, 8, 7, 14, 12, 6, 3, 10, 4, 5, 1, 2, 11]
p2    : [3, 6, 8, 10, 7, 12, 5, 13, 9, 11, 4, 2, 14, 1]
child : [5, 13, 9, 11, 14, 12, 6, 3, 10, 4, 2, 1, 8, 7]
meme multiset ? True


## 4. Mutation : swap ou inversion

**Attention au piege des deux probabilites.** Il y a *deux* tirages aleatoires distincts dans la mutation, qu'il ne faut pas confondre :

- **`mutation_rate` (defaut 0.2)** : probabilite **qu'un enfant subisse une mutation tout court**, decidee dans la boucle generationnelle juste apres le croisement.
- **`0.5` interne a `_mutate`** : une fois qu'on sait qu'un enfant *est* mute, c'est un pile-ou-face entre les deux operateurs (swap ou inversion). Ce **n'est pas** un taux de mutation.

**Pourquoi 0.2 et pas 0.05 ?** Le 0.05 (et meme la regle classique `1/L` ou L est la longueur du chromosome [4]) vient des GA sur **bitstrings**, ou la mutation s'applique **gene par gene** : chaque bit a une petite chance independante de flipper, donc sur 100 bits on attend en moyenne 5 modifications. Ici la mutation est **par individu** : avec `mutation_rate=0.2`, 20% des enfants subissent **une seule** modification (un swap ou une inversion). Sur une permutation de 50 clients, ca represente une perturbation tres locale - le taux par-gene equivalent serait minuscule. Les valeurs typiques pour les GA permutationnels (TSP, VRP) sont 0.1 a 0.3, comme rapporte par Eiben & Smith [4, chap. 4] et utilise par Prins [1, section 5] sur des instances VRP.

**Pourquoi deux operateurs ?**

- **Swap** : echange deux positions arbitraires. Casse jusqu'a 4 adjacences (2 autour de chaque gene). Perturbation **ponctuelle mais agressive**, utile pour sortir d'optima locaux.
- **Inversion** d'un segment `[i, j]` : retourne la sous-sequence. Ne casse que **2 adjacences** (les frontieres `i-1<->i` et `j<->j+1`), tout le reste du segment reste connecte. C'est exactement le mouvement **2-opt** introduit par Lin pour le TSP [5], donc une perturbation tres compatible avec le decodage Split.

Tirer a 50/50 entre les deux donne un melange diversification (swap) / exploitation prudente (inversion) sans avoir a regler un parametre de plus.


In [5]:
rng = random.Random(0)
for _ in range(3):
    muted = _mutate(clients[:], rng)
    print(muted)


[1, 2, 3, 4, 5, 6, 13, 12, 11, 10, 9, 8, 7, 14]
[1, 2, 3, 4, 5, 6, 7, 9, 8, 10, 11, 12, 13, 14]
[1, 2, 3, 4, 13, 6, 7, 8, 9, 10, 11, 12, 5, 14]


## 5. Boucle generationnelle complete

Une fois les briques en place, la generation est lineaire : selectionner deux parents par tournoi, croiser avec proba `crossover_rate`, muter avec proba `mutation_rate`, decoder par Split, evaluer, et recommencer jusqu'a remplir la nouvelle population. **L'elitisme** copie les `elitism` meilleurs individus tels quels dans la generation suivante : c'est une garantie **monotone** que le meilleur cout ne peut jamais regresser au fil des generations, ce qui evite qu'un mauvais croisement detruise la meilleure solution courante (resultat formel de De Jong [6]).

Les valeurs par defaut (`population_size=30`, `generations=100`, `crossover_rate=0.85`, `mutation_rate=0.2`, `tournament_k=3`, `elitism=2`) sont calquees sur les reglages recommandes par Prins [1, table 1] pour les GA-VRP, ajustes a la baisse pour rester dans des temps demo raisonnables. La cellule ci-dessous reduit `generations` a 50 pour la meme raison.


In [6]:
sol = genetic_algorithm(
    si,
    population_size=30,
    generations=50,
    tournament_k=3,
    crossover_rate=0.85,
    mutation_rate=0.2,
    elitism=2,
    memetic=True,
    seed=0,
)
print(f"cout = {sol.total_cost:.2f}  routes = {sol.route_count}")


cout = 1448.30  routes = 7


## 6. Effet du schema memetique

**Pourquoi un GA "pur" est-il insuffisant sur le VRP ?** Un GA explore l'espace par recombinaison, mais il n'a aucun mecanisme pour **descendre** vers l'optimum local le plus proche. Concretement, un enfant issu d'OX + mutation est une solution "plausible" mais quasi jamais localement optimale : il existe presque toujours un swap ou un 2-opt qui l'ameliore immediatement. Le GA pur passe alors son temps a comparer des solutions toutes mediocres entre elles, et la pression de selection ne suffit pas a reproduire systematiquement les bonnes adjacences. Cette faiblesse est documentee depuis longtemps (Goldberg [3], chap. 7).

**Le schema memetique** (terme du a Moscato [7]) corrige ca en intercalant une **descente locale** apres chaque decodage Split : chaque enfant devient un **optimum local** avant d'entrer dans la population. La selection compare donc des optima locaux, pas des solutions brutes. Consequences :

- **Qualite** : la borne basse de chaque generation augmente franchement (l'echantillon n'inclut plus de solutions trivialement ameliorables). C'est exactement le mecanisme exploite par Prins [1] qui combine GA + 2-opt + Or-opt.
- **Cout** : chaque evaluation passe d'un Split O(n^2) a un Split + recherche locale complete (relocate + swap + 2-opt jusqu'a stabilite). **Le ratio mesure ci-dessous sur n=15 est d'environ x30** (15 ms vs 489 ms) - c'est specifique a cette instance, le facteur depend de n et de la qualite des optima locaux atteints.
- **Compromis** : sans memetique, le GA est en general derriere SA et tabou ; avec memetique, il devient competitif voire meilleur sur les instances moyennes [1, table 4], au prix d'un budget temps superieur.

Le defaut du module est `memetic=True` parce que l'objectif du projet est la qualite de solution, pas le debit.


In [7]:
import time

for memetic in (False, True):
    t0 = time.perf_counter()
    sol = genetic_algorithm(si, population_size=30, generations=40, memetic=memetic, seed=0)
    dt = time.perf_counter() - t0
    print(f"memetic={memetic!s:<5} -> cout={sol.total_cost:.2f}  temps={dt*1000:.1f} ms")


memetic=False -> cout=1481.98  temps=15.8 ms


memetic=True  -> cout=1448.30  temps=489.2 ms


## 7. Tournoi et pression selective

**Pourquoi tournoi et pas roulette de fitness ?** La selection par roulette (probabilite proportionnelle au fitness) demande une **mise a l'echelle des couts** (un VRP minimise, donc il faut inverser ; et plus l'instance est petite, plus les couts sont resserres, ce qui rend la roulette quasi uniforme - probleme connu sous le nom de *fitness scaling problem*, Goldberg [3], chap. 4). Le tournoi evite tout ca : il ne depend que de l'**ordre relatif** des individus, pas de la valeur absolue de leur cout. C'est plus robuste, plus simple, et la pression selective se regle d'un seul parametre `k` - voir l'analyse formelle de Blickle & Thiele [8] sur les distributions de fitness apres tournoi.

**Effet de `k`** :

- `k=1` : selection purement aleatoire, aucune pression -> derive genetique (la population s'eloigne de l'optimum sans direction).
- `k=2` : tournoi doux, beaucoup de diversite preservee, convergence lente.
- `k=3` : compromis classique, recommande pour la majorite des problemes combinatoires (voir Eiben & Smith [4], section 5.2.2).
- `k>=5` : forte pression, convergence rapide mais **risque de convergence prematuree** (la population s'effondre autour d'un optimum local avant d'avoir pu explorer suffisamment).

**Limite de la demo ci-dessous** : sur n=15 avec `memetic=True`, l'instance est trop petite pour distinguer les valeurs de `k` - tous les runs convergent vers le meme optimum (1448.30). L'effet devient visible sur des instances plus grandes ou avec `memetic=False`. C'est aussi ce que confirme le notebook `benchmark.ipynb`.


In [8]:
for k in (2, 3, 5, 7):
    sol = genetic_algorithm(si, population_size=30, generations=40, tournament_k=k, seed=1)
    print(f"k={k} -> cout={sol.total_cost:.2f}")


k=2 -> cout=1448.30


k=3 -> cout=1448.30


k=5 -> cout=1448.30


k=7 -> cout=1448.30


## 8. Reproductibilite

Tous les tirages aleatoires (population initiale, tournoi, point de coupure OX, choix swap/inversion, paires mutees) passent par un unique `random.Random(seed)` cree au debut de l'algorithme. Aucun appel ne touche le `random` global. Consequence : a `seed` egal, l'algorithme produit **exactement** la meme trajectoire et donc la meme solution finale - propriete indispensable pour le debug et pour comparer equitablement plusieurs algorithmes dans `benchmark.py`.


In [9]:
a = genetic_algorithm(si, population_size=20, generations=20, seed=7)
b = genetic_algorithm(si, population_size=20, generations=20, seed=7)
print("identiques ?", a.total_cost == b.total_cost and a.routes == b.routes)


identiques ? True


## 9. En resume

Trois decisions structurelles font la specificite de cette implementation :

1. **Codage giant-tour + Split** [1] : on ne code pas une solution VRP, on code une permutation. Le decodeur Split (PD O(n^2) sur un DAG auxiliaire) garantit que chaque permutation est evaluee a son **meilleur** decoupage possible. C'est ce qui permet aux operateurs simples d'OX et de mutation de transmettre de l'information utile.
2. **OX et mutation par-individu** [2, 4] : choisis pour preserver la propriete de permutation par construction (pas de reparation post-hoc) et pour transmettre des **adjacences**, qui sont les schemas pertinents pour les problemes de tournees. Le `mutation_rate=0.2` reflete que la mutation est par-individu et non par-gene.
3. **Schema memetique** [7] : comble la faiblesse intrinseque du GA pur sur le VRP en garantissant que chaque individu evalue est un optimum local. Cout x30 sur cette instance, mais qualite competitive avec tabou et SA.

**Quand le choisir** : sur les instances moyennes a grandes ou quand on a un budget temps confortable. Pour des reponses quasi-instantanees ou des instances tres petites, GRASP ou SA sont generalement preferables.

---

## References

[1] **Prins, C. (2004).** *A simple and effective evolutionary algorithm for the vehicle routing problem.* Computers & Operations Research, 31(12), 1985-2002. https://doi.org/10.1016/S0305-0548(03)00158-8

[2] **Davis, L. (1985).** *Applying adaptive algorithms to epistatic domains.* Proceedings of the 9th International Joint Conference on Artificial Intelligence (IJCAI), 162-164.

[3] **Goldberg, D. E. (1989).** *Genetic Algorithms in Search, Optimization, and Machine Learning.* Addison-Wesley.

[4] **Eiben, A. E. & Smith, J. E. (2015).** *Introduction to Evolutionary Computing* (2nd ed.). Springer. https://doi.org/10.1007/978-3-662-44874-8

[5] **Lin, S. (1965).** *Computer solutions of the traveling salesman problem.* Bell System Technical Journal, 44(10), 2245-2269. https://doi.org/10.1002/j.1538-7305.1965.tb04146.x

[6] **De Jong, K. A. (1975).** *An Analysis of the Behavior of a Class of Genetic Adaptive Systems.* PhD thesis, University of Michigan.

[7] **Moscato, P. (1989).** *On evolution, search, optimization, genetic algorithms and martial arts: Towards memetic algorithms.* Caltech Concurrent Computation Program, Report 826.

[8] **Blickle, T. & Thiele, L. (1996).** *A comparison of selection schemes used in evolutionary algorithms.* Evolutionary Computation, 4(4), 361-394. https://doi.org/10.1162/evco.1996.4.4.361
